# **`Agent_01(Triage Agent)`**

In [ ]:
!pip install groq

In [ ]:
from google.colab import userdata
from groq import Groq

api_key = userdata.get('EGROQ_API_KEY')
client = Groq(api_key=api_key)
print("Connected!")

Connected!


In [ ]:
sample_email_subject="URGENT: Your account has been locked.Verify Now!"
sample_email_body="Dear customer,we have detected unusual activity on your account.Please verify your identity immediately by clicking the link below.Failure to do so within 24 hours may result in permanent aaccount suspension."

In [ ]:
def triage_agent(subject,body):
  prompt=f"""
  You are an email security triage agent.Analyze the following email and classify it as one of:Important,Spam, or Suspicious.

  Look for these patterns:
  -Urgency-based language(e.g,"act now", "24 hours","immediately")
  -Requests for sensitive information or account verification
  -Threats of account suspension or loss
  -Generic greetings instead of personalized ones.


Subject:{subject}
Body:{body}

Respond ONLY in this JSON format:
{{
"classification":"Important,Spam,or Suspicious",
"reasons":["reason1","reason2"],
"recommendation":"what the user should do"
}}
"""
  response=client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":prompt}]
  )
  return response.choices[0].message.content

#Test
result=triage_agent(sample_email_subject,sample_email_body)
print(result)
#

{
"classification":"Suspicious",
"reasons":["Urgency-based language", "Request for sensitive information (account verification)", 
           "Threat of account suspension (implying an urgent and coercive tone)"],
"recommendation":"Do not click on the link and verify account status through your regular login account. Contact customer support for assistance instead."
}


# **`TESTING (Important Email and Spam Email)`**

In [ ]:
# Important email test
important_subject = "Meeting rescheduled to 3 PM tomorrow"
important_body = "Hi team, our project sync meeting has been moved from 10 AM to 3 PM tomorrow due to a scheduling conflict. Please update your calendars. Thanks, Sarah"

result_important = triage_agent(important_subject, important_body)
print(result_important)

{
"classification":"Important",
"reasons":["Personalized greeting", "Specific subject matter (meeting reschedule)"],
"recommendation":"Mark the new meeting time in your calendar to avoid missing it."
}


In [ ]:
# Spam email test
spam_subject = "You've won a $1000 gift card!"
spam_body = "Congratulations! You have been randomly selected to receive a free $1000 Amazon gift card. Click here to claim your prize now before it expires!"

result_spam = triage_agent(spam_subject, spam_body)
print(result_spam)

{
"classification":"Spam",
"reasons":["Urgency-based language e.g. 'act now', 'immediately'", "Request for immediate action without clear explanation"],
"recommendation":"Do not click on the link and report the email as spam to prevent potential phishing threats."
}


##**` Agent_02(Phishing intelligence & Link Verification Agent)`**

In [ ]:
import re

def extract_urls(email_body):
    url_pattern = r'https?://[^\s]+'
    urls = re.findall(url_pattern, email_body)
    return urls

In [ ]:
def link_verification_agent(url, brand_context=""):
    prompt = f"""
    You are a phishing intelligence agent. Analyze this URL and determine
    if it is genuine or a phishing/scam attempt.

    Check for:
    - Brand impersonation (e.g., "amaz0n" instead of "amazon", extra words/dashes)
    - Suspicious domain patterns (unusual TLDs like .xyz, .top, misspellings)
    - Whether the URL matches the claimed sender/brand in the email context

    URL: {url}
    Email context: {brand_context}

    Respond ONLY in this JSON format:
    {{
        "verdict": "genuine/phishing",
        "trust_score": 0-100,
        "risk_level": "LOW/MEDIUM/HIGH",
        "reasons": ["reason1", "reason2"],
        "recommendation": "what the user should do"
    }}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

# **`Testing Agent_02`**

In [ ]:
# Test 1: Phishing URL
result1 = link_verification_agent(
    "http://amaz0n-offers.xyz/track",
    "Amazon Order Confirmation"
)
print("PHISHING TEST:")
print(result1)
print()

# Test 2: Genuine URL
result2 = link_verification_agent(
    "https://www.amazon.com/orders/track",
    "Amazon Order Confirmation"
)
print("GENUINE TEST:")
print(result2)

PHISHING TEST:
{
  "verdict": "phishing",
  "trust_score": 20,
  "risk_level": "HIGH",
  "reasons": [
    "Brand impersonation: 'amaz0n' instead of 'amazon'",
    "Suspicious domain TLD: '.xyz'",
    "Mismatched domain and email sender, indicating possible phishing attempt"
  ],
  "recommendation": "Do not click on the link, and instead, navigate to the official Amazon website directly and check your order status"
}

GENUINE TEST:
{
  "verdict": "genuine",
  "trust_score": 99,
  "risk_level": "LOW",
  "reasons": [
    "The URL exactly matches the claimed sender (Amazon)",
    "The domain pattern (.com) is a genuine Top Level Domain (TLD) for Amazon",
    "No suspicious characters or words are present in the URL"
  ],
  "recommendation": "You can safely click on this link and track your order on Amazon"
}


# **`Agent-to-Agent Communication: Full Pipeline (Agent 1 → Agent 2)`**

This section demonstrates the collaborative workflow between the Triage
Agent and the Link Verification Agent. Agent 1 classifies the incoming
email and passes the extracted context to Agent 2, which verifies any
embedded URLs. This structured hand-off between agents represents the
system's agent-to-agent communication protocols.


---




In [ ]:
def process_email(subject, body):
    # Agent 1: Triage
    triage_result = triage_agent(subject, body)
    print("=== AGENT 1 (Triage) OUTPUT ===")
    print(triage_result)

    # Extract URLs
    urls = extract_urls(body)

    # Agent 2: Only runs if there are links
    if urls:
        print("\n=== AGENT 2 (Link Verification) OUTPUT ===")
        for url in urls:
            link_result = link_verification_agent(url, subject)
            print(link_result)
    else:
        print("\nNo links found in this email. Skipping link verification.")


In [ ]:
sample_url_email_subject = "PayPal Security Alert"
sample_url_email_body = (
    "We detected unusual activity on your account. "
    "Verify your identity immediately at: http://paypa1-secure-login.xyz/verify"
)

label = "phishing"

In [ ]:
# Test the full pipeline
process_email(sample_url_email_subject, sample_url_email_body)

=== AGENT 1 (Triage) OUTPUT ===
{
"classification": "Suspicious",
"reasons": [
    "Urgency-based language ('verify your identity immediately')",
    "Request for sensitive information (account verification) with a suspicious link",
    "Generic greeting",
    "Unofficial PayPal notification (PayPal official notification would likely come from pay-pal.com)"
],
"recommendation": "Do not click on the link and instead, log in to your PayPal account directly at pay-pal.com to verify your account."
}

=== AGENT 2 (Link Verification) OUTPUT ===
{
  "verdict": "phishing",
  "trust_score": 20,
  "risk_level": "HIGH",
  "reasons": [
    "Brand impersonation: 'paypa1' is a misspelled version of 'PayPal'",
    "Suspicious domain pattern: '.xyz' is an unusual TLD, and the domain 'paypa1-secure-login' includes unnecessary words and dashes",
    "Mismatch between URL and claimed sender: The email claims to be from PayPal, but the URL does not match the expected PayPal login structure"
  ],
  "recomm

In [ ]:
def reflect_on_verdict(initial_result, url):
    prompt = f"""
    Review this phishing analysis and verify if the verdict is accurate.
    Confirm or correct the assessment if needed.

    Initial Analysis: {initial_result}
    URL: {url}

    Respond in the same JSON format as before.
    """
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Test
reflection_result = reflect_on_verdict(result1, "http://amaz0n-offers.xyz/track")
print(reflection_result)

**Phishing Analysis Verification**

    {
  "verdict": "phishing",
  "trust_score": 20,
  "risk_level": "HIGH",
  "reasons": [
    "Brand impersonation: 'amaz0n' instead of 'amazon'",
    "Suspicious domain TLD: '.xyz'",
    "Mismatched domain and email sender, indicating possible phishing attempt",
    "The URL contains misspelling and the TLD '.xyz' is not typical for official Amazon emails"
  ],
  "recommendation": "Do not click on the link, and instead, navigate to the official Amazon website directly and check your order status"
}

**Verdict Validation and Enhancement**

Upon reviewing the analysis, it is evident that the initial verdict of "phishing" is accurate. The reasons for this assessment include:

1.  **Brand impersonation:** The URL 'amaz0n-offers.xyz' attempts to mimic the official Amazon brand, with a deliberate misspelling ('0' instead of 'o') in the domain name.
2.  **Suspicious domain TLD:** The use of the '.xyz' top-level domain is not typical for official Amazon em

In [ ]:
!pip install openai

In [ ]:
from google.colab import userdata
from openai import OpenAI

openrouter_key = userdata.get('OPENROUTER_API_KEY')
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key
)
print("OpenRouter Connected!")

OpenRouter Connected!


In [ ]:
def reflect_on_verdict(initial_result, url):
    prompt = f"""
    Review this phishing analysis and verify if the verdict is accurate.
    Confirm or correct the assessment if needed.

    Initial Analysis: {initial_result}
    URL: {url}

    Respond in the same JSON format as before.
    """
    response = openrouter_client.chat.completions.create(
        model="meta-llama/llama-3.2-3b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
reflection_result = reflect_on_verdict(result1, "http://amaz0n-offers.xyz/track")
print(reflection_result)

Reviewing the phishing analysis:

{
  "verdict": "phishing",
  "trust_score": 20,
  "risk_level": "HIGH",
  "reasons": [
    "Brand impersonation: 'amaz0n' instead of 'amazon'",
    "Suspicious domain TLD: '.xyz'",
    "Mismatched domain and email sender, indicating possible phishing attempt"
  ],
  "recommendation": "Do not click on the link, and instead, navigate to the official Amazon website directly and check your order status"
}

Verification:
The analysis is mostly accurate, but I found some issues:

1. Brand impersonation: The red flags remain, but Amazon's brand name is misspelled as 'amaz0n', which is not a good practice. However, this alone doesn't necessarily mean it's a phishing attempt.
2. Suspicious domain TLD: The '.xyz' domain is a wildcard top-level domain, which can be used for phishing sites. However, it's not a definitive indicator of phishing.
3. Mismatched domain and email sender: While it's possible to misrepresent the sender's domain, the provided URL only chec